# JW $Z_4^{TF}$ system

Created: 03-09-2026

Objectives:
* Iterate on [this notebook](jw_z_4_tf_system.ipynb), using a more advanced optimization to find the projectors. Take 3 adjacent, disjoint regions $A, P, B$. For a given $\ket{v}$ supported in $P$, define $\rho_{AB} = \Tr_{(A \cup B)^c}\braket{v | \rho | v}$, and $\rho_A, \rho_B$ defined accordingly. We wish to find a $\ket{v}$ such that $\rho_{AB} = \rho_A \otimes \rho_B$. Thus the optimization norm we use is $$||\rho_{AB} - \rho_A \otimes \rho_B||_{Fr}$$, i.e. the trace norm squared of $\rho_{AB} - \rho_A \otimes \rho_B$. This reduces to $$\Tr(\rho_{AB}^2) + \Tr(\rho_A^2)\Tr(\rho_B^2) - 2\Tr(\rho(\rho_A \otimes \rho_B))$$
* Also need to ensure $\ket{v}$ respects fermion parity symmetry.

# Imports

In [1]:
import numpy as np

In [2]:
import jax
jax.config.update('jax_platform_name', 'cpu')

import jax.numpy as jnp

In [3]:
import matplotlib.pyplot as plt

In [4]:
from tqdm import tqdm

In [5]:
from functools import reduce
from operator import mul

In [6]:
from random import random

In [7]:
import quimb.tensor as qtn
import quimb as qu

In [8]:
from scipy.stats import unitary_group

In [9]:
from collections import Counter

In [10]:
import pandas as pd

In [11]:
from time import time

In [12]:
from humanize import naturalsize

# Definitions
## Construct cluster state

In [13]:
np_up_X_state = 1/(np.sqrt(2))*np.array([1,1])

In [14]:
qu_up_X_state = qtn.Tensor(
    data=np_up_X_state,
    inds=('k',),
    tags='prod'
)

In [15]:
np_CZ = np.diag([1,1,1,-1])

In [16]:
np_CZ = np_CZ.reshape((2,)*4)

In [17]:
qu_CZ = qtn.Tensor(
    data=np_CZ,
    inds=('k1', 'k2', 'b1', 'b2'),
    tags='CZ'
)

In [18]:
np_hadamard = np.pow(2, -1/2)*np.array([
    [1,1],
    [1,-1]
])

In [19]:
qu_hadamard = qtn.Tensor(data=np_hadamard, inds=('k', 'b'), tags='Had')

In [20]:
def get_cluster_state_qu_tensor_network(num_sites):
    assert (num_sites%2) == 0

    product_state_tensors = [
        qu_up_X_state.reindex({'k': f'kc_1_{i}'})
        for i in range(num_sites)
    ]

    first_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'kc_1_{i}',
            'b2': f'kc_1_{i+1}',
            'k1': f'kc_2_{i}',
            'k2': f'kc_2_{i+1}'
        })
        for i in range(0, num_sites, 2)
    ]


    second_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'kc_2_{i}',
            'b2': f'kc_2_{(i+1)%num_sites}',
            'k1': f'kh_{i}',
            'k2': f'k{(i+1)%num_sites}'
        })
        for i in range(1, num_sites+1, 2)
    ]

    hadamard_layer = [
        qu_hadamard.reindex({
            'b': f'kh_{i}',
            'k': f'k{i}'
        })
        for i in range(1, num_sites, 2)
    ]
    all_tensors = (
        product_state_tensors
        + first_layer_circuit_tensors
        + second_layer_circuit_tensors
        + hadamard_layer
    )

    out = qtn.TensorNetwork(all_tensors, virtual=True)
    out.mangle_inner_()

    return out

## Construct product state

In [21]:
np_up_X_state = 1/(np.sqrt(2))*np.array([1,1])

In [22]:
np_up_Z_state = np.array([1,0])

In [23]:
qu_up_Z_state = qtn.Tensor(
    data=np_up_Z_state,
    inds=('k',),
    tags='prod'
)

In [24]:
alternating_states = [
    qu_up_X_state,
    qu_up_Z_state
]

def get_product_qu_tensor_network(num_sites):
    assert (num_sites%2) == 0

    product_state_tensors = [
        alternating_states[i%2].reindex({'k': f'k{i}'})
        for i in range(num_sites)
    ]

    out = qtn.TensorNetwork(
        product_state_tensors,
        virtual=True
    )
    out.mangle_inner_()

    return out

## Symmetries

In [25]:
def multikron(arrays):
    return reduce(np.kron, arrays)

In [26]:
np_I = np.array([
    [1,0],
    [0,1]
])

np_X = np.array([
    [0,1],
    [1,0]
])

np_Y = np.array([
    [0,-1j],
    [1j,0]
])

np_Z = np.array([
    [1,0],
    [0,-1]
])

In [27]:
qu_I = qtn.Tensor(
    np_I,
    inds=['k', 'b'],
    tags='X'
)

qu_X = qtn.Tensor(
    np_X,
    inds=['k', 'b'],
    tags='X'
)

qu_Y = qtn.Tensor(
    np_Y,
    inds=['k', 'b'],
    tags='Y'
)

qu_Z = qtn.Tensor(
    np_Z,
    inds=['k', 'b'],
    tags='Z'
)

In [28]:
def get_multisite_qu_X(num_sites):
    np_many_X = multikron([np_X]*num_sites)

    out = qtn.Tensor(
        np_many_X,
        inds=['k', 'b'],
        tags='mulit_site_X',
    )

    return out

In [29]:
def get_multisite_qu_I(num_sites):
    np_many_I = multikron([np_I]*num_sites)

    out = qtn.Tensor(
        np_many_I,
        inds=['k', 'b'],
        tags='mulit_site_I',
    )

    return out

In [30]:
qu_spin_fermion_fp = (
    qu_I.reindex({'k': 'ks', 'b': 'bs'})
    & qu_Z.reindex({'k': 'kf', 'b': 'bf'})
).contract()

qu_unit_cell_fp = qu_spin_fermion_fp.fuse({
    'k': ['ks', 'kf'],
    'b': ['bs', 'bf']
})

In [31]:
"""
def get_multisite_qu_fp(num_sites):
    assert (num_sites%2)==0

    num_unit_cells = (num_sites//2)

    np
"""

'\ndef get_multisite_qu_fp(num_sites):\n    assert (num_sites%2)==0\n\n    num_unit_cells = (num_sites//2)\n\n    np\n'

Decompose T symmetry as $MK$:

In [32]:
np_00 = np.array([[1,0], [0,0]])
np_11 = np.array([[0,0], [0,1]])

In [33]:
def tensor_product_operators(op_1, op_2):
    out = (
        op_1[..., np.newaxis, np.newaxis]
        *op_2[np.newaxis, np.newaxis, ...]
    )

    return out

In [34]:
np_M = (
    tensor_product_operators(np_X, np_00)
    + tensor_product_operators(np_Y, np_11)
)

In [35]:
qu_M = qtn.Tensor(
    np_M,
    inds=['ks', 'bs', 'kf', 'bf']
)

In [36]:
np_M_reindexed = (
    qu_M
    .fuse({
        'k': ['ks', 'kf'],
        'b': ['bs', 'bf']
    })
    .transpose('k', 'b')
    .data
)

## Extracting projectors

In [37]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [38]:
def maximize_projector_states(rho, left_sites, proj_sites, right_sites):
    v = qtn.Tensor(
        data=random_uniform_complex((2,)*(len(proj_sites)-1)),
        inds=[f'k{i}' for i in proj_sites[:-1]]
    )

    tnopt = qtn.TNOptimizer(
        v,  # the tensor network we want to optimize
        loss_func,  # the function we want to minimize
        norm_fn=normalize_v,
        loss_constants={"rho": rho},
        loss_kwargs={
            "left_sites": left_sites,
            "proj_sites": proj_sites,
            "right_sites": right_sites,
        },
        autodiff_backend="jax",
        optimizer="L-BFGS-B",
        progbar=False
    )

    v_opt = tnopt.optimize(n=2000)

    embedded_v_opt = embed_fp_even_vector(v_opt).contract()

    return v_opt, embedded_v_opt, tnopt.losses

In [39]:
def projector_state_check(state, rho, left_sites):
    left_right_rho = (
        rho
        & state
        & state.conj().reindex({s: f'b{s[1:]}' for s in state.inds})
    )

    left_right_rho = left_right_rho.contract()

    left_inds = [
        f'{s}{i}'
        for i in left_sites
        for s in 'kb'
    ]

    schmidt_decomp = qtn.tensor_core.tensor_split(
        left_right_rho,
        left_inds=left_inds,
        method='svd',
        #cutoff=1e-6,
        cutoff_mode='abs',
        absorb=None,
        renorm=False,
        bond_ind='v'
    )

    schmidt_vals = schmidt_decomp.tensors[1]

    return schmidt_vals

### Embed vector

In [40]:
np_CX = (
    tensor_product_operators(np_00, np_I)
    + tensor_product_operators(np_11, np_X)
)

In [41]:
qu_CX = qtn.Tensor(
    np_CX,
    inds=['k1', 'b1', 'k2', 'b2']
)

In [42]:
np_up_Z_state = np.array([1,0])

In [43]:
qu_up_Z_state = qtn.Tensor(
    data=np_up_Z_state,
    inds=('k',),
    tags='Z0_pad'
)

In [44]:
def embed_fp_even_vector(v):
    # Take a vector of length 2N-1, and return a vector of length 2N
    # which commutes with IZIZ...IZIZ
    # Assuming v site ordering is spin-fermion-spin-...-fermion
    # I think we need at least two fermion sites for this to be reasonable

    sites = sorted(int(s[1:]) for s in v.inds)

    padded_site = sites[-1] + 1

    padded_v = (
        v
        & qu_up_Z_state.reindex({'k': f'k{padded_site}_0'})
    )
    num_cx_gates = len(sites)//2

    cx_gates = [
        qu_CX.reindex({
            'k1': f'k{sites[2*i+1]}',
            'b1': f'b{sites[2*i+1]}',
            'k2': f'k{padded_site}_{i+1}',
            'b2': f'k{padded_site}_{i}'
        })
        for i in range(num_cx_gates-1)
    ]

    i = num_cx_gates-1
    cx_gates.append(
        qu_CX.reindex({
            'k1': f'k{sites[2*i+1]}',
            'b1': f'b{sites[2*i+1]}',
            'k2': f'k{padded_site}',
            'b2': f'k{padded_site}_{i}'
        })
    )

    reindexed_padded_v = (
        padded_v
        .reindex({
            f'k{sites[2*i+1]}': f'b{sites[2*i+1]}'
            for i in range(num_cx_gates)
        })
    )
    sym_v = qtn.TensorNetwork([
        reindexed_padded_v,
        *cx_gates
    ])

    sym_v.mangle_inner_()

    return sym_v

### Loss function

In [45]:
def get_rho_purity(rho, sites):
    # Assuming rho is a Hermitian reduced density matrix
    reindex_map = (
        {f'k{i}': f'b{i}' for i in sites}
        | {f'b{i}': f'k{i}' for i in sites}
    )

    rho_other = rho.reindex(reindex_map)
    #rho_other.mangle_inner_()
    
    out = (rho & rho_other).contract()

    return out

In [46]:
def normalize_v(v):
    norm = (v & v.conj()).contract()
    w = v*jnp.power(norm, -0.5)
    return w

In [47]:
def loss_func(v, rho, left_sites, proj_sites, right_sites):
    embed_v = embed_fp_even_vector(v)
    
    rho_lr = (
        rho
        & embed_v.reindex({f'k{i}': f'b{i}' for i in proj_sites})
        & embed_v.conj()
    )
    rho_lr = rho_lr.contract()
    tr_rho_lr = (
        rho_lr
        .reindex({f'k{i}': f'b{i}' for i in left_sites + right_sites})
        .contract()
    )
    
    rho_l = rho_lr.reindex(
        {f'k{i}': f'b{i}' for i in right_sites}
    )
    rho_l = rho_l.contract()*jnp.power(tr_rho_lr, -0.5)
    
    rho_r = rho_lr.reindex(
        {f'k{i}': f'b{i}' for i in left_sites}
    )
    rho_r = rho_r.contract()*jnp.power(tr_rho_lr, -0.5)
    
    purity_lr = get_rho_purity(rho_lr, left_sites + right_sites)
    purity_l = get_rho_purity(rho_l, left_sites)
    purity_r = get_rho_purity(rho_r, right_sites)
    
    reindex_map = (
        {f'k{i}': f'b{i}' for i in left_sites+right_sites}
        | {f'b{i}': f'k{i}' for i in left_sites+right_sites}
    )
    
    cross_term = (
        rho_lr.reindex(reindex_map)
        & rho_l
        & rho_r
    )
    cross_term = cross_term.contract()

    out = jnp.real(
        (purity_lr + purity_l*purity_r - 2*cross_term)/
        purity_lr
    )

    return out

## Extract cut rho and EDM

In [48]:
def generate_edm_from_cut_state(cut_state, sites, num_defect_sites):
    # Lots of duplicate code, probably a better way to do this.
    assert 2*num_defect_sites < len(sites)
    assert (num_defect_sites%2) == 0
    assert (len(sites)%2) == 0
    assert (sites[0]%2) == 0

    left_defect_sites = sites[:num_defect_sites]
    right_defect_sites = sites[-num_defect_sites:]
    internal_sites = sites[num_defect_sites:-num_defect_sites]

    # Being sloppy with the gate indices as the symmetries are invariant
    # under transpose and conjugation.
    left_sym_gates = [
        qu_M.reindex({
            'ks': f'b{i}',
            'kf': f'b{i+1}',
            'bs': f'c{i}',
            'bf': f'c{i+1}'
        })
        for i in left_defect_sites[::2]
    ]

    inner_gates = [
        qu_M.reindex({
            'ks': f'k{i}',
            'kf': f'k{i+1}',
            'bs': f'b{i}',
            'bf': f'b{i+1}'
        })
        for i in internal_sites[::2]
    ]

    right_sym_gates = [
        qu_M.reindex({
            'ks': f'b{i}',
            'kf': f'b{i+1}',
            'bs': f'c{i}',
            'bf': f'c{i+1}'
        })
        for i in right_defect_sites[::2]
    ]

    reindex_map = (
        {
            f'k{i}': f'c{i}'
            for i in (left_defect_sites + right_defect_sites)
        }
        |
        {
            f'k{i}': f'b{i}'
            for i in internal_sites
        }
    )

    # The fact that we don't conjugate the reindexed cut_state means that
    # we are effectively implementing local complex conjugation.
    edm = (
        cut_state
        & cut_state.reindex(reindex_map)
        & left_sym_gates
        & inner_gates
        & right_sym_gates
    )

    edm = edm.contract()

    fuse_maps = [
        ('k_left', (f'k{i}' for i in left_defect_sites)),
        ('b_left', (f'b{i}' for i in left_defect_sites)),
        ('k_right', (f'k{i}' for i in right_defect_sites)),
        ('b_right', (f'b{i}' for i in right_defect_sites))
    ]

    edm.fuse(fuse_maps, inplace=True)

    return edm

## Defect operators

In [49]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [50]:
def solve_for_boundary_operators(edm, num_iters=100):
    # Careful, the indices are reversed here for ease.
    # i.e. the k, b indices have been swapped to make tensor contraction easier.
    scores = list()

    u_left = qtn.tensor_builder.rand_tensor(
        (edm.ind_size('b_left'), edm.ind_size('k_left')),
        inds=['k_left', 'b_left'],
        dtype='complex64'
    )

    u_right = qtn.tensor_builder.rand_tensor(
        (edm.ind_size('b_right'), edm.ind_size('k_right')),
        inds=['k_right', 'b_right'],
        dtype='complex64'
    )

    for _ in range(num_iters):
        right_edm = (edm & u_left).contract()
        data = right_edm.data
        U, S, VH = np.linalg.svd(data)
        scores.append(np.sum(S))
    
        sol = (U @ VH).conj().T
        u_right = qtn.Tensor(sol, inds = ['b_right', 'k_right'])

        left_edm = (edm & u_right).contract()
        data = left_edm.data
        U, S, VH = np.linalg.svd(data)
        scores.append(np.sum(S))
    
        sol = (U @ VH).conj().T
        u_left = qtn.Tensor(sol, inds = ['b_left', 'k_left'])

    return (u_left, u_right), scores

## Apply random unitary to groundstate

In [51]:
def generate_random_su2():
    # Randomly sample a unitary, and scale by the determinant.
    u = unitary_group.rvs(2)
    det_u = np.linalg.det(u)
    su = u*np.power(det_u, -0.5)

    return su

In [52]:
X =  generate_random_su2()

In [53]:
np.linalg.det(X)

np.complex128(0.9999999999999997-1.1102230246251562e-16j)

In [54]:
np.round(X @ (X.conj().T), 3)

array([[ 1.+0.j, -0.-0.j],
       [-0.+0.j,  1.+0.j]])

In [55]:
def generate_random_symmetry_respecting_unitary_no_offset():
    # Generate a unitary which commutes with MK, where
    # M = X tensor (|0><0|) + Y tensor (|1><1|)

    phi = np.random.uniform(0, 2*np.pi)
    phi_phasor = np.exp(1j*phi)
    u0 = np.diag([phi_phasor, phi_phasor.conj()])

    if random() > 0.5:
        u0 = u0 @ np_X

    u1 = generate_random_su2()

    u = (
        tensor_product_operators(u0, np_00)
        + tensor_product_operators(u1, np_11)
    )

    qu_u = qtn.Tensor(
        u,
        inds=['ks', 'bs', 'kf', 'bf']
    )
    
    return qu_u

In [56]:
def generate_random_symmetry_respecting_unitary_offset():
    # Generate a unitary U such that IUI commutes with (MM)K, where
    # M = X tensor (|0><0|) + Y tensor (|1><1|), and concatenation denotes
    # tensor product.
    phi = np.random.uniform(0, 2*np.pi)
    phi_phasor = np.exp(1j*phi)
    u0 = np.diag([phi_phasor, phi_phasor.conj()])

    phi = np.random.uniform(0, 2*np.pi)
    phi_phasor = np.exp(1j*phi)
    u1 = np.diag([phi_phasor, phi_phasor.conj()])

    u = (
        tensor_product_operators(np_00, u0)
        + tensor_product_operators(np_11, u1)
    )

    qu_u = qtn.Tensor(
        u,
        inds=['kf', 'bf', 'ks', 'bs']
    )
    
    return qu_u

In [57]:
def generate_random_symmetry_respecting_unitary(offset):
    if offset:
        return generate_random_symmetry_respecting_unitary_offset()
    else:
        return generate_random_symmetry_respecting_unitary_no_offset()

In [58]:
# Warning, likely making assupmtions about shape of psi, number of sites being even here etc.
def apply_haar_random_fdlu_to_quimb_state(psi, domains_dict):
    num_sites = domains_dict['num_system_sites']

    depth = domains_dict['fdlu_depth']
    offset = domains_dict['fdlu_offset']
    all_circuit_lists = [
        list() for _ in range(depth)
    ]

    for layer, circuit_list in enumerate(all_circuit_lists):
        delta = layer
        is_offset = ((offset + delta)%2 == 1)

        for i in range(num_sites//2):
            site_1 = ((2*i)+delta+offset)%num_sites
            site_2 = ((2*i)+1+delta+offset)%num_sites

            u = generate_random_symmetry_respecting_unitary(is_offset)

            if is_offset:
                reindex_map = {
                    'kf': f'k_{layer+1}_{site_1}',
                    'ks': f'k_{layer+1}_{site_2}',
                    'bf': f'k_{layer}_{site_1}',
                    'bs': f'k_{layer}_{site_2}'
                }
            else:
                reindex_map = {
                    'ks': f'k_{layer+1}_{site_1}',
                    'kf': f'k_{layer+1}_{site_2}',
                    'bs': f'k_{layer}_{site_1}',
                    'bf': f'k_{layer}_{site_2}'
                }
            
            qu_u = u.reindex(reindex_map)

            circuit_list.append(qu_u)

    all_tensors = (
        [psi.reindex({f'k{i}': f'k_0_{i}' for i in range(num_sites)})]
        + sum(all_circuit_lists, start=[])
    )

    out = (
        qtn
        .TensorNetwork(all_tensors, virtual=False)
        .mangle_inner_()
        .reindex({f'k_{depth}_{i}': f'k{i}' for i in range(num_sites)}) 
    )

    return out

In [59]:
def extract_time_reversal_information_after_random_fdlu(psi, domains_dict,
    num_random_states=20):

    out = list()

    for _ in range(num_random_states):
        rand_psi = apply_haar_random_fdlu_to_quimb_state(psi, domains_dict)
        out.append(extract_time_reversal_information(rand_psi, domains_dict))

    return out

In [60]:
def extract_factorization_time_reversal_information_after_random_fdlu(psi,
    domains_dict, num_random_states=20):
    out = list()

    for _ in range(num_random_states):
        rand_psi = apply_haar_random_fdlu_to_quimb_state(psi, domains_dict)
        data = extract_factorization_time_reversal_information(
            rand_psi,
            domains_dict
        )
        out.append(data)

    return out

In [61]:
def get_quimb_psi_from_quspin_psi(quspin_psi):
    quimb_psi = qtn.Tensor(
        quspin_psi[::-1].reshape((2,)*num_sites),
        inds=[f'k{i}' for i in range(num_sites)]
    )

    return quimb_psi

## Sweep function

In [62]:
def extract_projector(psi, rho_sites, num_pad_sites, jw_even=False):
    proj_sites = rho_sites
    left_sites = list(range(
        min(rho_sites) - num_pad_sites,
        min(rho_sites)
    ))
    right_sites = list(range(
        max(rho_sites)+1,
        max(rho_sites)+num_pad_sites+1
    ))
    all_sites = left_sites + proj_sites + right_sites
    rho = (psi & psi.conj().reindex({f'k{i}': f'b{i}' for i in all_sites}))
    raw_proj_vec, proj_vec, losses = maximize_projector_states(
        rho,
        left_sites,
        proj_sites,
        right_sites
    )

    schmidt_vals = projector_state_check(
        proj_vec,
        rho,
        left_sites
    )

    return raw_proj_vec, proj_vec, losses, schmidt_vals

In [63]:
def find_invariants_via_projectors_from_random_state(psi, domains_dict, jw_even=False):
    rand_psi = apply_haar_random_fdlu_to_quimb_state(psi, domains_dict)
    
    raw_left_proj_vec, left_proj_vec, *left_proj_vec_results = extract_projector(
        rand_psi,
        domains_dict['left_projector_sites'],
        domains_dict['num_projector_pad_sites'],
        jw_even
    )
    
    raw_right_proj_vec, right_proj_vec, *right_proj_vec_results = extract_projector(
        rand_psi,
        domains_dict['right_projector_sites'],
        domains_dict['num_projector_pad_sites'],
        jw_even
    )
    
    cut_sites = list(range(
        min(domains_dict['left_projector_sites']),
        max(domains_dict['right_projector_sites'])+1
    ))
    
    cut_rho_unprojected = (
        rand_psi
        & rand_psi.conj().reindex({f'k{i}': f'b{i}' for i in cut_sites})
    )
    
    left_proj_sites = domains_dict['left_projector_sites']
    right_proj_sites = domains_dict['right_projector_sites']
    
    cut_rho = (
        cut_rho_unprojected.reindex({
            **{f'k{i}': f'l{i}' for i in left_proj_sites},
            **{f'k{i}': f'l{i}' for i in right_proj_sites},
            **{f'b{i}': f'c{i}' for i in left_proj_sites},
            **{f'b{i}': f'c{i}' for i in right_proj_sites}
        })
        & left_proj_vec.conj().reindex({f'k{i}': f'l{i}' for i in left_proj_sites})
        & left_proj_vec
        & left_proj_vec.reindex({f'k{i}': f'c{i}' for i in left_proj_sites})
        & left_proj_vec.conj().reindex({f'k{i}': f'b{i}' for i in left_proj_sites})
        & right_proj_vec.conj().reindex({f'k{i}': f'l{i}' for i in right_proj_sites})
        & right_proj_vec
        & right_proj_vec.reindex({f'k{i}': f'c{i}' for i in right_proj_sites})
        & right_proj_vec.conj().reindex({f'k{i}': f'b{i}' for i in right_proj_sites})
    )
    
    cut_rho_trace = (
        cut_rho
        .reindex({f'b{i}': f'k{i}' for i in cut_sites})
        .contract()
    )
    
    cut_rho = cut_rho/cut_rho_trace
    
    tranpose_map = (
        {f'k{i}': f'b{i}' for i in cut_sites}
        | {f'b{i}': f'k{i}' for i in cut_sites}
    )
    
    cut_rho_purity = (
        (cut_rho & cut_rho.reindex(tranpose_map))
        .contract()
    )
    
    sub_cut_sites = list(range(
        max(domains_dict['left_projector_sites'])+1,
        min(domains_dict['right_projector_sites'])
    ))
    
    sub_cut_rho = (
        cut_rho
        .reindex({f'k{i}': f'b{i}' for i in left_proj_sites + right_proj_sites})
        .contract()
    )
    
    sub_cut_rho_trace = (
        sub_cut_rho
        .reindex({f'b{i}': f'k{i}' for i in sub_cut_sites})
        .contract()
    )
    
    tranpose_map = (
        {f'k{i}': f'b{i}' for i in sub_cut_sites}
        | {f'b{i}': f'k{i}' for i in sub_cut_sites}
    )
    
    sub_cut_rho_purity = (
        (sub_cut_rho & sub_cut_rho.reindex(tranpose_map))
        .contract()
    )
    
    cut_score, cut_state, cut_overlap = get_dominant_eigenvector(sub_cut_rho, False)

    cut_state_fermion_parity = compute_fermion_parity(cut_state, sub_cut_sites)

    edm = generate_edm_from_cut_state(
        cut_state,
        sub_cut_sites,
        domains_dict['num_defect_sites']
    )

    defect_ops_results = solve_for_boundary_operators(
        edm,
        num_iters=20
    )

    left_defect_sites = list(range(
        max(domains_dict['left_projector_sites']) + 1,
        max(domains_dict['left_projector_sites']) + 1 + domains_dict['num_defect_sites']
    ))
    right_defect_sites = list(range(
        min(domains_dict['right_projector_sites']) - domains_dict['num_defect_sites'],
        min(domains_dict['right_projector_sites'])
    ))
    
    left_rdm = (
        cut_rho
        .reindex({
            f'b{i}': f'k{i}'
            for i in cut_sites if i not in left_defect_sites
        })
    )
    left_rdm = left_rdm.contract()
    left_fuse_map = [
        ('k_left', [f'k{i}' for i in left_defect_sites]),
        ('b_left', [f'b{i}' for i in left_defect_sites])
    ]
    left_rdm.fuse(left_fuse_map, inplace=True)
    
    right_rdm = (
        cut_rho
        .reindex({
            f'b{i}': f'k{i}'
            for i in cut_sites if i not in right_defect_sites
        })
    )
    right_rdm = right_rdm.contract()
    right_fuse_map = [
        ('k_right', [f'k{i}' for i in right_defect_sites]),
        ('b_right', [f'b{i}' for i in right_defect_sites])
    ]
    right_rdm.fuse(right_fuse_map, inplace=True)

    left_defect_op, right_defect_op = defect_ops_results[0]

    np_left_rdm = (
        left_rdm
        .transpose('k_left', 'b_left')
        .data
    )

    np_left_defect_op = (
        left_defect_op
        .transpose('k_left', 'b_left')
        .data
        .T
    )

    fp_list = [np_I, np_Z]
    np_fp = multikron([
        fp_list[i%2]
        for i in range(domains_dict['num_defect_sites'])
    ])

    left_defect_op_invariant = np.trace(
        np_fp
        @ np_left_defect_op.conj().T
        @ np_fp
        @ np_left_defect_op
        @ np_left_rdm
    )

    np_right_rdm = (
        right_rdm
        .transpose('k_right', 'b_right')
        .data
    )
    
    np_right_defect_op = (
        right_defect_op
        .transpose('k_right', 'b_right')
        .data
        .T
    )

    right_defect_op_invariant = np.trace(
        np_fp
        @ np_right_defect_op.conj().T
        @ np_fp
        @ np_right_defect_op
        @ np_right_rdm
    )

    out = {
        'left_proj_vec': left_proj_vec,
        'raw_left_proj_vec': raw_left_proj_vec,
        'left_proj_vec_results': left_proj_vec_results,
        'right_proj_vec': right_proj_vec,
        'raw_right_proj_vec': raw_right_proj_vec,
        'right_proj_vec_results': right_proj_vec_results,
        'cut_rho_trace': cut_rho_trace,
        'cut_rho_purity': cut_rho_purity,
        'sub_cut_rho_trace': sub_cut_rho_trace,
        'sub_cut_rho_purity': sub_cut_rho_purity,
        'cut_score': cut_score,
        'cut_state': cut_state,
        'cut_state_fermion_parity': cut_state_fermion_parity,
        'cut_overlap': cut_overlap,
        'defect_ops_scores': defect_ops_results[1],
        'left_defect_op': left_defect_op,
        'right_defect_op': right_defect_op,
        'left_defect_op_invariant': left_defect_op_invariant,
        'right_defect_op_invariant': right_defect_op_invariant,
        #'sub_cut_rho': sub_cut_rho,
        #'cut_rho_unprojected': cut_rho_unprojected
    }

    return out

## Find dominant eigenvector

In [64]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [65]:
def lanczos_iteration(rho, v, sites):
    w = (
        rho & v.reindex({f'k{i}': f'b{i}' for i in sites})
    ).contract()

    w_norm = np.sqrt((w & w.conj()).contract())

    out = w/w_norm

    return out

In [66]:
def multiply_state_by_jw_string(state, sites):
    gates = [
        qu_Z.reindex({'k': f'k{i}', 'b': f'b{i}'})
        for i in sites if (i%2 == 1)
    ]

    out = qtn.TensorNetwork(
        [
            *gates,
            state.reindex({f'k{i}': f'b{i}' for i in sites if (i%2 == 1)})
        ]
    )

    return out.contract()

In [67]:
def lanczos_iteration_jw_even(rho, v, sites):
    w = multiply_state_by_jw_string(v, sites)
    w = (v+w)/2

    w = (
        rho & w.reindex({f'k{i}': f'b{i}' for i in sites})
    ).contract()

    w = (multiply_state_by_jw_string(w, sites) + w)/2

    w_norm = np.sqrt((w & w.conj()).contract())

    out = w/w_norm

    return out

In [68]:
def lanczos_algorithm(rho, v, sites, num_iters=20, jw_even=False):
    update_func = lanczos_iteration_jw_even if jw_even else lanczos_iteration
    for _ in range(num_iters):
        v = update_func(rho, v, sites)

    return v

In [69]:
def get_dominant_eigenvector(rho, jw_even=False):
    k_inds = [i for i in rho.inds if i.startswith('k')]
    #b_inds = [i for i in rho.inds if i.startswith('b')]

    sites = [int(s[1:]) for s in k_inds]

    v = qtn.Tensor(
        data=random_uniform_complex((2,)*len(sites)),
        inds=[f'k{i}' for i in sites]
    )

    v = lanczos_algorithm(
        rho,
        v,
        sites,
        num_iters=20,
        jw_even=jw_even
    )

    update_func = lanczos_iteration_jw_even if jw_even else lanczos_iteration
    new_v = update_func(rho, v, sites)

    overlap = np.abs((v & new_v.conj()).contract())

    score = (
        v.reindex({f'k{i}': f'b{i}' for i in sites})
        & rho
        & v.conj()
    )
    score = score.contract()


    return score, v, overlap

In [70]:
def compute_fermion_parity(state, sites):
    odd_sites = [i for i in sites if (i%2 == 1)]

    gates = [
        qu_Z.reindex({'k': f'k{i}', 'b': f'b{i}'})
        for i in odd_sites
    ]

    gate_exp = (
        state.reindex({f'k{i}': f'b{i}' for i in odd_sites})
        & gates
        & state.conj()
    ).contract()

    return gate_exp

# Check random unitaries and symmetry

In [71]:
domains_dict = {
    'num_system_sites': 24,
    'left_projector_sites': list(range(4, 8)),
    'right_projector_sites': list(range(16, 20)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 2,
    'fdlu_offset': 1
}

In [72]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [73]:
rand_psi = apply_haar_random_fdlu_to_quimb_state(cluster_psi, domains_dict)

In [74]:
(rand_psi & rand_psi.conj()).contract()

np.complex128(0.999999999999997-2.7755575615628914e-17j)

In [75]:
(rand_psi & cluster_psi.conj()).contract()

np.complex128(0.00016197692891849874-6.776263578034403e-21j)

In [76]:
symmetry_gates = [
    qu_M.reindex({
        'ks': f'k{i}', 'bs':f'b{i}',
        'kf': f'k{i+1}', 'bf':f'b{i+1}',
    })
    for i in range(0, domains_dict['num_system_sites'], 2)
]

In [77]:
sym_cluster_psi = qtn.TensorNetwork(
    [
        rand_psi.reindex({f'k{i}': f'b{i}' for i in range(domains_dict['num_system_sites'])}),
        *symmetry_gates
    ]
)

In [78]:
(
    sym_cluster_psi & sym_cluster_psi.conj()
).contract()

np.complex128(0.9999999999999969-4.85722573273506e-17j)

In [79]:
(
    sym_cluster_psi & rand_psi.conj()
).contract()

np.complex128(-0.0006212087541048538+1.1858461261560205e-19j)

In [80]:
(
    sym_cluster_psi & rand_psi
).contract()

np.complex128(0.9999999999999971+3.885780586188048e-16j)

In [81]:
product_psi = get_product_qu_tensor_network(domains_dict['num_system_sites'])

In [82]:
rand_psi = apply_haar_random_fdlu_to_quimb_state(product_psi, domains_dict)

In [83]:
(rand_psi & rand_psi.conj()).contract()

np.complex128(0.999999999999998+5.551115123125783e-17j)

In [84]:
(rand_psi & product_psi.conj()).contract()

np.complex128(0.0005569656226113008-5.747227057618769e-20j)

In [85]:
symmetry_gates = [
    qu_M.reindex({
        'ks': f'k{i}', 'bs':f'b{i}',
        'kf': f'k{i+1}', 'bf':f'b{i+1}',
    })
    for i in range(0, domains_dict['num_system_sites'], 2)
]

In [86]:
sym_product_psi = qtn.TensorNetwork(
    [
        rand_psi.reindex({f'k{i}': f'b{i}' for i in range(domains_dict['num_system_sites'])}),
        *symmetry_gates
    ]
)

In [87]:
(
    sym_product_psi & sym_product_psi.conj()
).contract()

np.complex128(0.999999999999998+5.551115123125783e-17j)

In [88]:
(
    sym_product_psi & rand_psi.conj()
).contract()

np.complex128(-1.7734021330183284e-07+1.2281977735187355e-20j)

In [89]:
(
    sym_product_psi & rand_psi
).contract()

np.complex128(0.999999999999998+5.551115123125783e-17j)

So random unitaries are symmetric, nice.

# Test - Cluster state

In [90]:
domains_dict = {
    'num_system_sites': 24,
    'left_projector_sites': list(range(4, 8)),
    'right_projector_sites': list(range(16, 20)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 0,
    'fdlu_offset': 0
}

In [91]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [92]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(
        cluster_psi,
        domains_dict,
        jw_even=True
    )
    results.append(current)

100%|█████████████████████████████████████████████| 20/20 [00:14<00:00,  1.42it/s]


### Analyze results

#### Projector scores

In [93]:
np.round(np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([ 0.,  0., -0., -0.,  0., -0., -0., -0.,  0.,  0., -0.,  0.,  0.,
        0.,  0.,  0., -0., -0., -0., -0.])

In [94]:
np.round(np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([-0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0., -0., -0.,  0.,
       -0., -0.,  0.,  0., -0., -0.,  0.])

In [95]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [96]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.])

In [97]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.])

#### Purities

In [98]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j])

In [99]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j])

In [100]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j])

Purities are the same, which one could likely prove.

In [101]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j])

#### Cut state results

In [102]:
np.array(
    [d['cut_state_fermion_parity'] for d in results]
)

array([-1.+4.85344544e-18j, -1.+1.35932848e-17j, -1.+1.99766483e-17j,
       -1.+2.99748848e-18j,  1.-1.42449247e-17j, -1.+1.42145727e-17j,
        1.+6.78337980e-19j,  1.+8.87954661e-18j,  1.-1.85171801e-17j,
        1.+1.75644615e-17j, -1.+1.50226792e-17j,  1.-1.21818815e-17j,
       -1.-5.31886490e-18j, -1.-1.19149615e-17j,  1.+1.19486474e-17j,
       -1.+5.06569690e-18j,  1.-1.29988599e-17j, -1.+4.79330003e-18j,
       -1.+1.07624936e-17j, -1.-9.84343923e-18j])

So the FP of the cut state can be even or odd. Interesting!

In [103]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

#### Defect op scores overlaps

In [104]:
np.round(np.array([d['defect_ops_scores'][-1] for d in results]), 5)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [105]:
np.array([d['defect_ops_scores'][-1] for d in results])

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [106]:
[d['defect_ops_scores'][-1] for d in results]

[np.float64(0.9999999999999996),
 np.float64(0.9999999999999998),
 np.float64(1.0),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000002),
 np.float64(1.0),
 np.float64(0.9999999999999999),
 np.float64(0.9999999999999998),
 np.float64(1.0000000000000009),
 np.float64(0.9999999999999999),
 np.float64(1.0000000000000004),
 np.float64(0.9999999999999999),
 np.float64(0.9999999999999998),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0000000000000004),
 np.float64(1.0),
 np.float64(0.9999999999999998),
 np.float64(0.9999999999999998),
 np.float64(0.9999999999999998)]

#### Phases

In [107]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [108]:
left_phases.shape

(20,)

In [109]:
np.round(left_phases, 3)

array([-1.+0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.-0.j, -1.-0.j,
       -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.-0.j,
       -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.-0.j])

Weird things happening when cut_score = 0. Why is this happening?

In [110]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [111]:
right_phases.shape

(20,)

In [112]:
np.round(right_phases, 3)

array([-1.+0.j, -1.+0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.-0.j,
       -1.+0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.-0.j, -1.-0.j, -1.+0.j,
       -1.-0.j, -1.+0.j, -1.-0.j, -1.-0.j, -1.-0.j, -1.-0.j])

# Test - Cluster state - depth 1

In [113]:
domains_dict = {
    'num_system_sites': 32,
    'left_projector_sites': list(range(4, 10)),
    'right_projector_sites': list(range(22, 28)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 1,
    'fdlu_offset': 0
}

In [114]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [115]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(
        cluster_psi,
        domains_dict,
        jw_even=True
    )
    results.append(current)

100%|█████████████████████████████████████████████| 20/20 [01:17<00:00,  3.85s/it]


### Analyze results

#### Projector scores

In [116]:
np.round(np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([ 0.,  0., -0., -0., -0.,  0., -0., -0., -0.,  0.,  0., -0.,  0.,
       -0.,  0., -0., -0.,  0.,  0.,  0.])

In [117]:
np.round(np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([-0.,  0.,  0., -0.,  0., -0., -0.,  0.,  0.,  0., -0., -0., -0.,
       -0., -0., -0., -0., -0.,  0., -0.])

In [118]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [119]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0.52 , 0.455, 0.215, 0.471, 0.556, 0.315, 0.551, 0.312, 0.365,
       0.308, 0.313, 0.619, 0.454, 0.418, 0.639, 0.294, 0.727, 0.756,
       0.356, 0.42 ])

In [120]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0.619, 0.52 , 0.459, 0.673, 0.945, 0.901, 0.459, 0.939, 0.325,
       0.71 , 0.371, 0.79 , 0.545, 0.836, 0.703, 0.771, 0.325, 0.891,
       0.519, 0.262])

#### Purities

In [121]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j])

In [122]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j])

In [123]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j])

Purities are the same, which one could likely prove.

In [124]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j])

#### Cut state results

In [125]:
np.array(
    [d['cut_state_fermion_parity'] for d in results]
)

array([ 1.-1.99756131e-18j, -1.-1.63191018e-18j,  1.-1.86613000e-18j,
       -1.+5.65191236e-19j,  1.-3.40894894e-18j,  1.+7.89679126e-19j,
       -1.+6.89167143e-19j,  1.+1.66822645e-18j, -1.-2.08030391e-18j,
       -1.-1.32201975e-18j, -1.-1.07316474e-18j, -1.+7.97046086e-19j,
       -1.-4.83253342e-19j,  1.-9.89940686e-19j, -1.+2.30256736e-18j,
        1.+2.02321657e-18j,  1.-6.21468710e-19j,  1.-2.66942960e-18j,
        1.+7.16534633e-19j,  1.-2.38371973e-18j])

So the FP of the cut state can be even or odd. Interesting!

In [126]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

#### Defect op scores overlaps

In [127]:
np.round(np.array([d['defect_ops_scores'][-1] for d in results]), 5)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [128]:
np.array([d['defect_ops_scores'][-1] for d in results])

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [129]:
[d['defect_ops_scores'][-1] for d in results]

[np.float64(0.9999999999999996),
 np.float64(1.0000000000000004),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0000000000000004),
 np.float64(0.9999999999999996),
 np.float64(1.0),
 np.float64(0.9999999999999996),
 np.float64(0.9999999999999998),
 np.float64(1.0000000000000002),
 np.float64(1.0),
 np.float64(1.0000000000000004),
 np.float64(1.0000000000000002),
 np.float64(0.9999999999999998),
 np.float64(1.0000000000000004),
 np.float64(0.9999999999999999),
 np.float64(1.0000000000000004),
 np.float64(1.0000000000000004),
 np.float64(0.9999999999999993)]

#### Phases

In [130]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [131]:
left_phases.shape

(20,)

In [132]:
np.round(left_phases, 3)

array([-1.-0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.+0.j,
       -1.-0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j,
       -1.-0.j, -1.-0.j, -1.+0.j, -1.-0.j, -1.-0.j, -1.+0.j])

Weird things happening when cut_score = 0. Why is this happening?

In [133]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [134]:
right_phases.shape

(20,)

In [135]:
np.round(right_phases, 3)

array([-1.-0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.+0.j,
       -1.-0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j,
       -1.+0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.-0.j, -1.-0.j])

# Test - Cluster state - depth 2

In [136]:
domains_dict = {
    'num_system_sites': 32,
    'left_projector_sites': list(range(4, 10)),
    'right_projector_sites': list(range(22, 28)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 2,
    'fdlu_offset': 0
}

In [137]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [138]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(
        cluster_psi,
        domains_dict,
        jw_even=True
    )
    results.append(current)

100%|█████████████████████████████████████████████| 20/20 [03:27<00:00, 10.36s/it]


### Analyze results

#### Projector scores

In [139]:
np.round(np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([ 0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0., -0.,  0.,  0.,
        0.,  0.,  0.,  0., -0.,  0.,  0.])

In [140]:
np.round(np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([ 0.,  0.,  0.,  0.,  0.,  0., -0.,  0., -0.,  0., -0., -0., -0.,
        0.,  0.,  0.,  0., -0., -0.,  0.])

In [141]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [142]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0.05 , 0.564, 0.285, 0.193, 0.43 , 0.314, 0.772, 0.187, 0.673,
       0.04 , 0.289, 0.02 , 0.35 , 0.533, 0.536, 0.124, 0.517, 0.562,
       0.748, 0.135])

In [143]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0.54 , 0.337, 0.57 , 0.213, 0.146, 0.132, 0.216, 0.888, 0.233,
       0.838, 0.057, 0.754, 0.721, 0.256, 0.678, 0.238, 0.724, 0.14 ,
       0.03 , 0.149])

#### Purities

In [144]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.   -0.j, 1.   -0.j, 1.   -0.j, 1.   +0.j, 1.   -0.j, 1.   -0.j,
       1.   -0.j, 1.   +0.j, 1.   -0.j, 1.   +0.j, 0.999-0.j, 1.   -0.j,
       1.   -0.j, 0.996+0.j, 1.   -0.j, 1.   -0.j, 0.999+0.j, 0.999-0.j,
       1.   +0.j, 1.   -0.j])

In [145]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j])

In [146]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.   +0.j, 1.   +0.j, 1.   +0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j,
       1.   +0.j, 1.   +0.j, 1.   -0.j, 1.   +0.j, 0.999-0.j, 1.   +0.j,
       1.   -0.j, 0.996+0.j, 1.   -0.j, 1.   -0.j, 0.999-0.j, 0.999-0.j,
       1.   +0.j, 1.   -0.j])

Purities are the same, which one could likely prove.

In [147]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.   +0.j, 1.   +0.j, 1.   +0.j, 1.   -0.j, 1.   -0.j, 1.   -0.j,
       1.   +0.j, 1.   +0.j, 1.   -0.j, 1.   +0.j, 0.999-0.j, 1.   +0.j,
       1.   -0.j, 0.996+0.j, 1.   -0.j, 1.   -0.j, 0.999-0.j, 0.999-0.j,
       1.   +0.j, 1.   -0.j])

#### Cut state results

In [148]:
np.array(
    [d['cut_state_fermion_parity'] for d in results]
)

array([ 1.-1.27344288e-18j, -1.+2.37652476e-19j, -1.-4.35601277e-18j,
       -1.+2.42170753e-18j, -1.+2.44059114e-18j, -1.-1.63691756e-19j,
       -1.+1.30224818e-18j,  1.-6.70633134e-19j,  1.+3.31155420e-18j,
       -1.-2.06694072e-20j,  1.+1.41816436e-18j, -1.-1.63301413e-19j,
        1.-2.68110723e-18j, -1.-2.30637286e-20j, -1.-1.04214913e-18j,
        1.+1.40175626e-18j, -1.-2.35390253e-18j, -1.+5.79303836e-19j,
        1.+1.43980891e-18j, -1.+5.55393259e-19j])

So the FP of the cut state can be even or odd. Interesting!

In [149]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

#### Defect op scores overlaps

In [150]:
np.round(np.array([d['defect_ops_scores'][-1] for d in results]), 5)

array([0.32436, 0.60354, 0.66803, 0.76421, 0.45304, 0.70655, 0.80017,
       0.46442, 0.79968, 0.72163, 0.51106, 0.16705, 0.72755, 0.87378,
       0.5216 , 0.61846, 0.02351, 0.73611, 0.10203, 0.65014])

In [151]:
np.array([d['defect_ops_scores'][-1] for d in results])

array([0.32435665, 0.60353688, 0.6680312 , 0.76421012, 0.4530402 ,
       0.70655385, 0.80017377, 0.46442307, 0.79968482, 0.7216265 ,
       0.51106161, 0.1670518 , 0.72754853, 0.87377891, 0.52160358,
       0.618461  , 0.0235144 , 0.73610532, 0.1020262 , 0.65013706])

In [152]:
[d['defect_ops_scores'][-1] for d in results]

[np.float64(0.32435664984462914),
 np.float64(0.6035368802887705),
 np.float64(0.6680312026620716),
 np.float64(0.7642101246520334),
 np.float64(0.45304019619802105),
 np.float64(0.7065538493470069),
 np.float64(0.8001737677876509),
 np.float64(0.4644230720804807),
 np.float64(0.7996848220603622),
 np.float64(0.7216265008276403),
 np.float64(0.5110616148735007),
 np.float64(0.16705179949825913),
 np.float64(0.7275485313278706),
 np.float64(0.873778912567873),
 np.float64(0.5216035790471826),
 np.float64(0.6184609971599364),
 np.float64(0.023514399201502893),
 np.float64(0.736105316341883),
 np.float64(0.10202619536533528),
 np.float64(0.6501370620742076)]

#### Phases

In [153]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [154]:
left_phases.shape

(20,)

In [155]:
np.round(left_phases, 3)

array([-1.   +0.j, -1.   +0.j, -1.   +0.j, -1.   +0.j, -1.   +0.j,
       -1.   +0.j, -1.   -0.j, -1.   -0.j, -1.   +0.j, -1.   -0.j,
       -1.   +0.j, -1.   -0.j, -1.   -0.j, -0.998-0.j, -1.   +0.j,
       -1.   +0.j, -1.   -0.j, -1.   +0.j, -1.   -0.j, -1.   +0.j])

Weird things happening when cut_score = 0. Why is this happening?

In [156]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [157]:
right_phases.shape

(20,)

In [158]:
np.round(right_phases, 3)

array([-0.966+0.j, -0.935+0.j, -0.993-0.j, -0.999-0.j, -0.89 +0.j,
       -0.993+0.j, -0.668-0.j, -0.983-0.j, -0.812-0.j, -0.961-0.j,
       -0.989+0.j, -0.963+0.j, -0.967+0.j, -0.969-0.j, -1.   +0.j,
       -0.736+0.j, -0.816-0.j, -0.655-0.j, -0.072-0.j, -0.967+0.j])

Need to bump up defect size, but otherwise looking very good.

# Test - Product state - depth 2

In [159]:
domains_dict = {
    'num_system_sites': 32,
    'left_projector_sites': list(range(4, 10)),
    'right_projector_sites': list(range(22, 28)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 2,
    'fdlu_offset': 0
}

In [160]:
product_psi = get_product_qu_tensor_network(domains_dict['num_system_sites'])

In [161]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(
        product_psi,
        domains_dict,
        jw_even=True
    )
    results.append(current)

100%|█████████████████████████████████████████████| 20/20 [01:11<00:00,  3.58s/it]


### Analyze results

#### Projector scores

In [162]:
np.round(np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([ 0., -0., -0., -0., -0.,  0.,  0., -0., -0.,  0., -0., -0., -0.,
       -0., -0., -0.,  0., -0., -0., -0.])

In [163]:
np.round(np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([-0., -0., -0., -0.,  0., -0.,  0., -0.,  0., -0.,  0., -0., -0.,
       -0., -0., -0.,  0.,  0., -0., -0.])

In [164]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [165]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [166]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

#### Purities

In [167]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j])

In [168]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j])

In [169]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j])

Purities are the same, which one could likely prove.

In [170]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j])

#### Cut state results

In [171]:
np.array(
    [d['cut_state_fermion_parity'] for d in results]
)

array([1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])

So the FP of the cut state can be even or odd. Interesting!

In [172]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

#### Defect op scores overlaps

In [173]:
np.round(np.array([d['defect_ops_scores'][-1] for d in results]), 5)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [174]:
np.array([d['defect_ops_scores'][-1] for d in results])

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [175]:
[d['defect_ops_scores'][-1] for d in results]

[np.float64(1.0000000000000002),
 np.float64(1.0000000000000002),
 np.float64(1.0),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000009),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000002),
 np.float64(0.9999999999999997),
 np.float64(0.9999999999999998),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000002),
 np.float64(1.0),
 np.float64(1.0000000000000002),
 np.float64(0.9999999999999999),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000002),
 np.float64(1.0),
 np.float64(0.9999999999999997),
 np.float64(1.0000000000000009)]

#### Phases

In [176]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [177]:
left_phases.shape

(20,)

In [178]:
np.round(left_phases, 3)

array([1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j])

Weird things happening when cut_score = 0. Why is this happening?

In [179]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [180]:
right_phases.shape

(20,)

In [ ]:
np.round(right_phases, 3)